# ML Hyperparameter Tuning - Grid Search

This notebook demonstrates hyperparameter tuning for TempCNN using `ml_tune_grid`, which performs exhaustive grid search over all parameter combinations.

For random search (more efficient for large search spaces), see `05_ml_tuning_random.ipynb`.

Results are written to `tuning_results.json`.

## Runtime (why it can be slow)

Each **cell in the grid** runs a **full training** (all `epochs` you set) on the training sample set. Cost is roughly:

`#combinations × time_one_tempcnn_run`

So `batch_size` × `epochs` with 2×2 values = **4 full trainings**. Deep models also **retry on CPU** if GPU memory is tight, which multiplies wall time.

**Ways to keep demos fast:** use **fewer / smaller grids**, **lower `epochs`**, `verbose: false`, keep **`cv: 0`** (single hold-out; k-fold multiplies work). For production tuning, prefer **`ml_tune_random`** with a modest `n_iter` and bounded `epochs` (see `05_ml_tuning_random.ipynb`).

In [ ]:
import openeo # type: ignore

In [ ]:
connection = openeo.connect(url="http://127.0.0.1:8000")
connection.authenticate_basic("brian", "123456")

In [ ]:
training_set = "https://github.com/e-sensing/sitsdata/raw/main/data/samples_deforestation_rondonia.rds"

In [ ]:
tempcnn_model_init = connection.mlm_class_tempcnn(
    optimizer="adam",
    learning_rate=0.0005,
    seed=42,
)

In [ ]:
# Default below is a small grid for shorter demos (2 combinations).
# For a serious search, widen epochs / batch_size or add commented axes — each
# new value multiplies total training runs.

param_grid = {
    # "cnn_layers": [[64, 64, 64], [128, 128, 128]],
    # "cnn_kernels": [[5, 5, 5], [7, 7, 7]],
    # "learning_rate": [0.0005, 0.0001],
    "batch_size": [32, 64],
    "epochs": [20, 40],  # e.g. [50, 100] is much slower (4+ min per combo is common)
}

In [ ]:
process_graph = {
    "tempcnn1": {
        "process_id": "mlm_class_tempcnn",
        "arguments": {
            "optimizer": "adam",
            "learning_rate": 0.0005,
            "seed": 42,
            "verbose": True
        },
    },
    "tune1": {
        "process_id": "ml_tune_grid",
        "arguments": {
            "model": {"from_node": "tempcnn1"},
            "training_data": training_set,
            "target": "label",
            "parameters": param_grid,
            "scoring": "accuracy",
            "cv": 0,
            "seed": 42
        },
        "result": True,
    },
}

job = connection.create_job(
    process_graph=process_graph,
    title="TempCNN hyperparameter tuning (grid search)",
    description="Exhaustive grid search for TempCNN hyperparameters",
)
job.start_and_wait()
results = job.get_results()

In [ ]:
results.download_files("data/outputs_grid/")